# s1p31 analysis
Warehouse fulfilment: daily operations, service levels and margin. Every section below runs the same functions as `main.py`.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
root = Path.cwd().resolve()
while not (root / "config" / "settings.py").exists():
    if root.parent == root:
        raise SystemExit("Open this notebook from inside the backend folder")
    root = root.parent
sys.path.insert(0, str(root))
import pandas as pd
from src.utils.paths import P, ensure_dirs
from src.utils.io_utils import load_frame, enriched
from src.data.loading import load_inputs
from src.data.cleaning import clean_data
from src.features.build import build_features, add_target
from src.analysis.explore import run_charts
from src.models.primary_model import fit_primary_model
from src.models.segments import run_segments
from src.models.forecasting import fit_margin_regression, build_forecast
from src.analytics.rules import apply_rules
from src.analytics.what_if import build_scenarios
from src.analytics.goals import ensure_goals, evaluate_goals
from src.analytics.findings import build_insights
from src.reporting.excel_export import write_workbook
from src.reporting.dashboard import write_dashboard, print_summary
ensure_dirs()

## Data intake
Read the file, match the headers, check the schema, and remove any personal-data columns.

In [ ]:
raw = load_inputs(verbose=True)
raw.head()

## Cleaning
Types, duplicates, impossible values and extreme operating outliers are handled here. The row counts before and after are kept for the record.

In [ ]:
clean = clean_data(raw, verbose=True)
clean.describe().T.head(15)

## Features
Ratios and flags a warehouse manager can read directly, plus an operational risk score and the low-margin target (the weakest quarter of site-days by margin).

In [ ]:
if 'clean' not in globals():
    clean = load_frame('clean')
feat = add_target(build_features(clean, verbose=True))
feat[['low_margin', 'ops_risk_score']].describe()

## Exploratory charts
Each chart answers one business question.

In [ ]:
if 'feat' not in globals():
    feat = enriched()
run_charts(feat)

## Low-margin risk model
Three or more models are compared with stratified cross-validation and scored once on a held-out test set. Operating measures only; revenue and cost lines are excluded so the model cannot read the answer from the accounts.

In [ ]:
feat = fit_primary_model(feat, verbose=True)

## Segments and unusual days

In [ ]:
feat = run_segments(feat, verbose=True)

## Margin regression and revenue forecast
The forecast uses a chronological holdout and is compared with a seasonal-naive baseline.

In [ ]:
reg = fit_margin_regression(feat, verbose=True)
fc = build_forecast(feat, verbose=True)

## Alerts, priorities and actions

In [ ]:
feat = apply_rules(feat, verbose=True)

## What-if scenarios and goals

In [ ]:
scen = build_scenarios(feat, verbose=True)
ensure_goals()
progress = evaluate_goals(feat, verbose=True)

## Insights

In [ ]:
insights = build_insights(feat, verbose=True)

## Workbook, dashboard and summary

In [ ]:
write_workbook(feat)
write_dashboard(feat)
print_summary(feat)

Next: run `python export_for_frontend.py` from the backend folder to refresh the dashboard data in `output/`.